In [ ]:
import sys
import os
from pathlib import Path  # noqa: F401

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

import numpy as np  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402
import seaborn as sns  # noqa: E402
import mne  # noqa: E402

from src.preprocessing.pipeline import DatasetHandler  # noqa: E402
from src.filtering.dataset_filter import DatasetFilter  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    ExperimentNames,
    CoordinateSystems,
    PreprocessedDataVariants,
    MusicTypeVariants,
    ConditionVariants,
    ExclusionCategories,
    SingleDataMetadata,
)
from src.visualization.preprocessing_plots import DatasetPlotter  # noqa: E402
from src.definitions.constants import ProjectPaths  # noqa: E402

%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
mne.set_log_level("ERROR")
print("Setup complete.")

# Time Alignment Analysis

This notebook analyses and verifies the **time alignment** of EEG recordings across
participants using the TAG (stimulus marker) channel.

Capabilities:
* Run cross-correlation-based time alignment across participants
* Plot cross-correlation curves for each participant pair vs. the reference
* Overlay all TAG music signals to verify alignment
* Compare aligned vs. unaligned signals visually
* Plot the pairwise correlation heatmap of aligned TAG signals
* **Per-participant Global Field Power overlay** per condition, reusing the
  already-aligned (`cropped`) recordings and highlighting amplitude outliers

> **Parameters to tweak:** `CONDITION`, `MUSIC_TYPE`, `EXCLUSION_CATEGORIES`,
> `PLOT_WINDOW_SEC`, `CROSSCORR_MAX_LAG_SEC`, `REUSE_ALIGNMENT` in the
> *Configuration* cell below.

## Configuration

In [ ]:
# ── Experiment selection ──────────────────────────────────────────────────────
CONDITION = ConditionVariants.PLACEBO
MUSIC_TYPE = MusicTypeVariants.CLASSICAL
EXCLUSION_CATEGORIES = [ExclusionCategories.BAD_MUSIC]

# ── Time alignment parameters ─────────────────────────────────────────────────
# Set to True to actually crop and save aligned data after alignment
CROP_AND_SAVE = False

# ── Reuse / compute ───────────────────────────────────────────────────────────
# When True and a complete set of `cropped` recordings exists for a group, the GFP
# overlay reuses them instead of re-running the alignment + crop (mirrors the
# stimulus-alignment notebook). When the cache is missing it falls back to running
# the time alignment and cropping, then saving the result.
REUSE_ALIGNMENT = True

# ── Visualisation parameters ──────────────────────────────────────────────────
PLOT_WINDOW_SEC = 10.0  # seconds of signal to show in the overlap plot
PLOT_T_START_SEC = 0.0  # start time (s) for the overlap plot
CROSSCORR_MAX_LAG_SEC = 5.0  # ±lag range (s) for the cross-correlation plots

# ── GFP overlay ───────────────────────────────────────────────────────────────
CONDITIONS_TO_PLOT = [ConditionVariants.PLACEBO, ConditionVariants.PSILOCYBIN]
OUTLIER_Z_THRESH = 3.5  # modified z-score cutoff on peak GFP (µV)

# ── Plot saving ──────────────────────────────────────────────────────────────
SAVE_PLOTS = True
PLOTS_DIR = ProjectPaths.NOTEBOOKS_DIR / "00-preprocessing" / "plots" / "time_alignment"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
_plot_suffix = f"{CONDITION.value}_{MUSIC_TYPE.value}"
print(f"Plots will be saved to: {PLOTS_DIR}")
print(f"Reuse cached alignment: {REUSE_ALIGNMENT}")

## Dataset Initialisation

In [ ]:
dataset_handler = DatasetHandler(
    ExperimentNames.PSILO_MUSIC, CoordinateSystems.HYDROGEL_257_NO_FIDUCIALS
)
dataset_metadata = dataset_handler.dataset_metadata
print(f"Dataset size: {len(dataset_metadata)} recordings")

## Run Time Alignment

Align all recordings for the selected condition and music type using cross-correlation
of the TAG (stimulus marker) signals.

In [ ]:
aligned_signals, time_aligner = dataset_handler.align_time_series(
    MUSIC_TYPE,
    CONDITION,
    EXCLUSION_CATEGORIES,
    plot_alignment_results=False,
    data_type_to_load=PreprocessedDataVariants.RAW_AFTER_ICA,
)
print(f"Aligned {len(aligned_signals)} recordings")
if time_aligner is not None:
    print(f"Reference index: {time_aligner.reference_idx}")
    print(f"Shifts (samples): {time_aligner.shifts}")

## Cross-Correlation Plots

Plot the cross-correlation function for each recording vs. the reference to inspect
the quality and uniqueness of the alignment peak.

In [ ]:
if aligned_signals and time_aligner is not None:
    ref_idx = time_aligner.reference_idx
    ref_signal = aligned_signals[ref_idx]
    sfreq = time_aligner.sfreq

    for i, sig in enumerate(aligned_signals):
        if i == ref_idx:
            continue
        save_path = (
            str(PLOTS_DIR / f"crosscorr_{_plot_suffix}_vs_ref_idx{i}.png")
            if SAVE_PLOTS
            else ""
        )
        DatasetPlotter.plot_crosscorr_vs_shift(
            ref_signal,
            sig,
            sfreq=sfreq,
            max_lag_sec=CROSSCORR_MAX_LAG_SEC,
            label1=f"Reference (idx {ref_idx})",
            label2=f"Recording {i}",
            save_fig=save_path,
        )
else:
    print("No aligned signals available — check configuration.")

## Aligned TAG Signal Overlay

After alignment, load the cropped TAG signals and plot all overlaid on the same axis
to verify synchrony.

In [ ]:
filtered_df = DatasetFilter.filter_dataset_by_all_categories(
    dataset_handler.dataset_metadata,
    dataset_handler.excluded_participants_metadata,
    [MUSIC_TYPE],
    [CONDITION],
    EXCLUSION_CATEGORIES,
)

all_tags, sfreq = dataset_handler._load_all_tags(
    filtered_df, PreprocessedDataVariants.RAW_AFTER_ICA
)
tag_signals = [tag.tag_signal for tag in all_tags]
print(f"Loaded {len(tag_signals)} TAG signals, sampling rate: {sfreq} Hz")

In [ ]:
# Plot the aligned TAG signals via DatasetHandler helper
# aligned_signals from align_time_series() are already cropped to the overlap window
if aligned_signals and time_aligner is not None:
    dataset_handler.plot_aligned_tags(aligned_signals, time_aligner)
else:
    print("No aligned signals available — run the alignment cell first.")

## Signal Overlap Plot

Show a short time window of all aligned TAG signals overlaid for detailed inspection.

In [ ]:
if tag_signals:
    # After alignment, derive the aligned (shifted) signals for overlay
    if time_aligner is not None:
        shifts = time_aligner.shifts
        max_shift = max(abs(s) for s in shifts)
        aligned_tags = [
            sig[max_shift + s : max_shift + s + (len(sig) - 2 * max_shift)]
            for sig, s in zip(tag_signals, shifts)
        ]
    else:
        aligned_tags = tag_signals

    DatasetPlotter.plot_signal_overlap(
        aligned_tags,
        sfreq=sfreq,
        t_start=PLOT_T_START_SEC,
        time_duration=PLOT_WINDOW_SEC,
        save_fig=str(PLOTS_DIR / f"signal_overlap_{_plot_suffix}.png")
        if SAVE_PLOTS
        else "",
    )
else:
    print("No TAG signals available.")

## Apply Alignment (Crop and Save)

Set `CROP_AND_SAVE = True` in the Configuration cell to crop all recordings to the
common aligned time window and save them to disk.

In [ ]:
if CROP_AND_SAVE:
    if time_aligner is None:
        raise RuntimeError("time_aligner is None — run the alignment cell first.")
    dataset_handler.crop_all_raw_to_alignment(time_aligner)
    print("All recordings cropped and saved.")
else:
    print("CROP_AND_SAVE is False — skipping. Set it to True to write files to disk.")

## Per-Participant Signal Overlay (Global Field Power)

Per-participant signal over time, overlaid on a common time axis — one line per
participant, one subplot per **(music type, condition)** group (CLASSIC and
PSYTRANCE × Placebo and Psilocybin).

Following the project-standard format, each channel is first **z-scored in time per
participant**, then the per-participant line is the **Global Field Power** of those
z-scored channels (std across channels per timepoint), so the lines are on a
comparable scale (no further normalization).

Two variants are plotted (**without** vs **with** the excluded participants), and in
*both* the cell independently **suggests participants for exclusion** as amplitude
outliers (robust modified z-score on peak GFP > `OUTLIER_Z_THRESH`). Each highlighted
line is labelled with its **exclusion reason** from the metadata (bad_music,
artifacts, …) and/or its suggested peak.

Highlighted lines are coloured by **class (hue family)** and given a distinct
**shade + line style per individual** within that class, so you can read the class
*and* tell the individuals apart (the legend names each):

- 🟢 **green family** — excluded **and** an amplitude outlier → exclusion validated.
- 🔴 **red family** — an amplitude outlier but **not** excluded → possible *missed* exclusion.
- 🟠 **orange family** — excluded but **not** an amplitude outlier → excluded for another
  reason (shown in the label, e.g. `bad_music`); expected, not amplitude-driven.
- grey — kept, unremarkable.

> **Note:** time-alignment exclusions are mostly `bad_music` (an unusable TAG
> channel), so the excluded recordings **cannot be TAG-aligned**. In the "with
> excluded" variant the included recordings are shown from their aligned `cropped`
> data, while the excluded ones are shown from a plain crop of `after_ica` from the
> recording start (**unaligned, best effort**, marked in the label) — compare
> amplitude, not timing.

In [ ]:
# ── Per-participant Global Field Power (z-scored channels) ────────────────────
# Each channel is z-scored in time per participant (the project-standard format),
# then GFP = std across the z-scored channels per timepoint gives one line per
# participant. One subplot per (music type, condition) group. Two variants per run
# (WITHOUT vs WITH excluded participants); both flag amplitude outliers as exclusion
# suggestions and label highlighted lines with their exclusion reason. Highlighted
# lines are coloured by CLASS (hue family) with a distinct SHADE + line style per
# individual: green = excluded & outlier (validated), red = outlier & NOT excluded
# (possible miss), orange = excluded for a non-amplitude reason. Excluded recordings
# are usually `bad_music` (unusable TAG) and cannot be TAG-aligned, so in the WITH
# variant they are a plain crop of after_ica from the recording start (unaligned,
# best effort, marked in the label); included recordings use their aligned `cropped`.
MUSIC_TYPES_TO_PLOT = [MusicTypeVariants.CLASSICAL, MusicTypeVariants.PSYTRANCE]
CONDITIONS_TO_PLOT = [ConditionVariants.PLACEBO, ConditionVariants.PSILOCYBIN]
OUTLIER_Z_THRESH = 3.5  # modified z-score cutoff on peak GFP

CLASS_CMAPS = {"validated": plt.cm.Greens, "missed": plt.cm.Reds, "other": plt.cm.Oranges}
CLASS_LINESTYLES = ["-", "--", "-.", ":"]


def _channel_gfp(raw):
    """GFP after z-scoring each channel in time (per participant)."""
    data = raw.get_data(picks="eeg")  # (n_channels, n_times)
    z = (data - data.mean(axis=1, keepdims=True)) / (
        data.std(axis=1, keepdims=True) + 1e-12
    )
    return z.std(axis=0)  # (n_times,) — std across the z-scored channels


def _group_df(music_type, condition, exclusion_categories):
    return DatasetFilter.filter_dataset_by_all_categories(
        dataset_handler.dataset_metadata,
        dataset_handler.excluded_participants_metadata,
        [music_type],
        [condition],
        exclusion_categories,
    )


def exclusion_reasons(music_type, condition):
    """Map filename / participant -> exclusion reason(s) from the metadata.

    A blank condition or music type in the metadata acts as a wildcard (applies to
    all). Returns (by_filename, by_participant) dicts of reason strings.
    """
    em = dataset_handler.excluded_participants_metadata
    cond = em[SingleDataMetadata.CONDITION.value].fillna("").str.strip()
    music = em[SingleDataMetadata.MUSIC_TYPE.value].fillna("").str.strip()
    keep = ((cond == "") | (cond == condition.value)) & (
        (music == "") | (music == music_type.value)
    )
    by_file, by_pid = {}, {}
    for _, r in em[keep].iterrows():
        reason = str(r[SingleDataMetadata.EXCLUSION_EXPLANATION.value]).strip()
        fname = str(r[SingleDataMetadata.FILENAME.value]).strip()
        pid = str(r[SingleDataMetadata.PARTICIPANT_ID.value]).strip()
        if fname and fname.lower() != "nan":
            by_file.setdefault(fname, set()).add(reason)
        else:
            by_pid.setdefault(pid, set()).add(reason)
    return by_file, by_pid


def reason_for(filename, pid, by_file, by_pid):
    reasons = by_file.get(filename, set()) | by_pid.get(pid, set())
    return "/".join(sorted(reasons))


def included_aligned_gfp(music_type, condition):
    """Included group (EXCLUSION_CATEGORIES applied): aligned `cropped` GFP.

    Reuses the cache when complete (streamed per recording); otherwise re-runs the
    TAG alignment for the included group and crops in memory (no disk writes).
    """
    fdf = _group_df(music_type, condition, EXCLUSION_CATEGORIES)
    filenames = fdf[SingleDataMetadata.FILENAME].tolist()
    labels = fdf[SingleDataMetadata.PARTICIPANT_ID].astype(str).tolist()
    paths = [
        dataset_handler.get_preprocessing_results_path(
            f.split(".")[0], PreprocessedDataVariants.RAW_CROPPED
        )
        for f in filenames
    ]
    if REUSE_ALIGNMENT and len(paths) > 0 and all(p.exists() for p in paths):
        gfp, sfreq = [], None
        for f in filenames:
            raw = dataset_handler.load_data_file(
                f, is_processed=True,
                processed_data_type=PreprocessedDataVariants.RAW_CROPPED, preload=True,
            )
            sfreq = raw.info["sfreq"]
            gfp.append(_channel_gfp(raw))
            del raw
        return labels, filenames, gfp, sfreq
    # Recompute the included-group alignment in memory (TAG is fine here).
    _, ta = dataset_handler.align_time_series(
        music_type, condition, EXCLUSION_CATEGORIES,
        data_type_to_load=PreprocessedDataVariants.RAW_AFTER_ICA,
    )
    sfreq = ta.sfreq
    pid_by_file = dict(zip(filenames, labels))
    labels, filenames, gfp = [], [], []
    for tag_object, shift in zip(ta.all_tags, ta.shifts):
        f = tag_object.filename
        raw = dataset_handler.load_data_file(
            f, is_processed=True,
            processed_data_type=PreprocessedDataVariants.RAW_AFTER_ICA, preload=True,
        )
        cs, ce = ta.get_crop_indices_for_signal(shift)
        raw.crop(tmin=cs / sfreq, tmax=ce / sfreq)
        gfp.append(_channel_gfp(raw))
        labels.append(pid_by_file.get(f, f))
        filenames.append(f)
        del raw
    return labels, filenames, gfp, sfreq


def excluded_unaligned_gfp(music_type, condition, target_len):
    """Would-be-excluded recordings: raw crop of after_ica from the start (unaligned).

    Their TAG is typically unusable, so they cannot be aligned; we crop to the
    included aligned length purely to overlay comparable signals.
    """
    full = _group_df(music_type, condition, [])
    included = set(
        _group_df(music_type, condition, EXCLUSION_CATEGORIES)[SingleDataMetadata.FILENAME]
    )
    rows = full[~full[SingleDataMetadata.FILENAME].isin(included)]
    labels, filenames, gfp = [], [], []
    for _, row in rows.iterrows():
        f = row[SingleDataMetadata.FILENAME]
        raw = dataset_handler.load_data_file(
            f, is_processed=True,
            processed_data_type=PreprocessedDataVariants.RAW_AFTER_ICA, preload=True,
        )
        if raw.n_times > target_len:
            raw.crop(tmax=(target_len - 1) / raw.info["sfreq"])
        gfp.append(_channel_gfp(raw))
        labels.append(str(row[SingleDataMetadata.PARTICIPANT_ID]))
        filenames.append(f)
        del raw
    return labels, filenames, gfp


def amplitude_outliers(peak_gfp, thresh):
    """Indices whose peak GFP is a high outlier by robust modified z-score."""
    med = np.median(peak_gfp)
    mad = np.median(np.abs(peak_gfp - med))
    if mad == 0:  # degenerate spread -> fall back to std
        mod_z = (peak_gfp - peak_gfp.mean()) / (peak_gfp.std() + 1e-12)
    else:
        mod_z = 0.6745 * (peak_gfp - med) / mad
    return set(np.where(mod_z > thresh)[0].tolist())


def _styles_by_class(class_of):
    """Per-index (color, linestyle): hue family by class, distinct shade per member."""
    members = {}
    for i, cls in class_of.items():
        members.setdefault(cls, []).append(i)
    style = {}
    for cls, idxs in members.items():
        cmap = CLASS_CMAPS[cls]
        for k, i in enumerate(idxs):
            frac = 0.7 if len(idxs) == 1 else 0.55 + 0.4 * k / (len(idxs) - 1)
            style[i] = (cmap(frac), CLASS_LINESTYLES[k % len(CLASS_LINESTYLES)])
    return style


def plot_gfp_variant(drop_excluded):
    """One figure, one subplot per (music type, condition). ``drop_excluded`` toggles."""
    variant = "without excluded" if drop_excluded else "with excluded"
    groups = [(m, c) for m in MUSIC_TYPES_TO_PLOT for c in CONDITIONS_TO_PLOT]
    fig, axes = plt.subplots(len(groups), 1, figsize=(15, 5 * len(groups)), squeeze=False)
    for ax, (music_type, condition) in zip(axes[:, 0], groups):
        labels, filenames, gfp, sfreq = included_aligned_gfp(music_type, condition)
        excluded_now = [False] * len(gfp)
        if not drop_excluded:
            target_len = min(len(g) for g in gfp)
            e_labels, e_files, e_gfp = excluded_unaligned_gfp(
                music_type, condition, target_len
            )
            labels += e_labels
            filenames += e_files
            gfp += e_gfp
            excluded_now += [True] * len(e_gfp)

        peak_gfp = np.array([g.max() for g in gfp])
        n = min(len(g) for g in gfp)
        times = np.arange(n) / sfreq

        by_file, by_pid = exclusion_reasons(music_type, condition)
        outliers = amplitude_outliers(peak_gfp, OUTLIER_Z_THRESH)

        class_of = {}
        for i in range(len(gfp)):
            is_out = i in outliers
            if is_out and excluded_now[i]:
                class_of[i] = "validated"
            elif is_out:
                class_of[i] = "missed"
            elif excluded_now[i]:
                class_of[i] = "other"
        style = _styles_by_class(class_of)

        for i in range(len(gfp)):
            if i not in style:
                ax.plot(times, gfp[i][:n], lw=0.6, alpha=0.4, color="0.6")

        excl_report, suggest_report, missed_report = [], [], []
        for i in sorted(style):
            is_out = i in outliers
            reason = reason_for(filenames[i], labels[i], by_file, by_pid)
            parts = []
            if excluded_now[i]:
                parts.append(f"excluded: {reason or 'yes'} (unaligned)")
                excl_report.append(f"{labels[i]}({reason or '?'})")
            elif reason:
                parts.append(f"flagged: {reason}")
            if is_out:
                parts.append(f"suggest peak {peak_gfp[i]:.2f}")
                suggest_report.append(labels[i])
                if not excluded_now[i]:
                    missed_report.append(labels[i])
            color, ls = style[i]
            ax.plot(times, gfp[i][:n], lw=1.8, alpha=0.95, color=color, ls=ls, zorder=5,
                    label=f"{labels[i]} — {', '.join(parts)}")

        ax.set_title(
            f"Per-participant GFP (z-scored channels, {variant}) — "
            f"{music_type.value} / {condition.value} (n={len(gfp)})"
        )
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("GFP (z-scored channels)")
        if style:
            ax.legend(loc="upper right", ncol=2, fontsize=7, framealpha=0.6)
        print(f"[{variant}] {music_type.value}/{condition.value}: "
              f"excluded={excl_report or 'none'} | "
              f"suggested(outliers)={suggest_report or 'none'} | "
              f"outlier-not-excluded={missed_report or 'none'}")

    plt.tight_layout()
    if SAVE_PLOTS:
        suffix = "without_excluded" if drop_excluded else "with_excluded"
        fig.savefig(PLOTS_DIR / f"participant_gfp_overlay_{suffix}.png", dpi=150)
    plt.show()


plot_gfp_variant(drop_excluded=True)  # without excluded participants
plot_gfp_variant(drop_excluded=False)  # with excluded participants

## Bad-Music Exclusion Check (TAG signal)

`bad_music` exclusions mean the **TAG (music-marker) channel** is wrong, which the
EEG GFP overlay above cannot reveal — so this cell verifies those exclusions on the
TAG channel itself. For each (music type, condition) group that has `bad_music`
exclusions, the TAG signals of the **included** recordings are drawn thin/grey (the
expected music-marker waveform) and the `bad_music`-**excluded** recordings are drawn
bold/red (distinct shade per participant, named in the legend).

If the exclusion is reasonable the red TAGs should look clearly different from the
grey ones — flat, saturated, noise-like, or otherwise lacking the marker structure
the included recordings share. Signals are shown raw (not aligned); set
`TAG_SHOW_SEC` to zoom into the first *N* seconds.

In [ ]:
# ── Bad-music TAG comparison ──────────────────────────────────────────────────
# Overlay the TAG (music-marker) signal of bad_music-excluded recordings (red,
# one shade each) against the included recordings (grey) per (music, condition)
# group, so the exclusion can be sanity-checked on the channel it concerns.
TAG_SHOW_SEC = None  # set to e.g. 60.0 to zoom into the first N seconds


def bad_music_filenames(music_type, condition):
    """Filenames excluded specifically for `bad_music` in this group."""
    em = dataset_handler.excluded_participants_metadata
    cond = em[SingleDataMetadata.CONDITION.value].fillna("").str.strip()
    music = em[SingleDataMetadata.MUSIC_TYPE.value].fillna("").str.strip()
    expl = em[SingleDataMetadata.EXCLUSION_EXPLANATION.value].fillna("").str.strip()
    keep = (
        (expl == ExclusionCategories.BAD_MUSIC.value)
        & ((cond == "") | (cond == condition.value))
        & ((music == "") | (music == music_type.value))
    )
    return em[keep][SingleDataMetadata.FILENAME.value].dropna().tolist()


def load_tags(filenames):
    """Load TAG signals for the given filenames; returns (TAGObjects, sfreq)."""
    meta = dataset_handler.dataset_metadata
    sub = meta[meta[SingleDataMetadata.FILENAME].isin(filenames)]
    if len(sub) == 0:
        return [], None
    return dataset_handler._load_all_tags(sub, PreprocessedDataVariants.RAW_AFTER_ICA)


_pid_by_file = dict(
    zip(
        dataset_handler.dataset_metadata[SingleDataMetadata.FILENAME],
        dataset_handler.dataset_metadata[SingleDataMetadata.PARTICIPANT_ID].astype(str),
    )
)

# Only build panels for groups that actually have bad_music exclusions.
panels = [
    (m, c)
    for m in MUSIC_TYPES_TO_PLOT
    for c in CONDITIONS_TO_PLOT
    if bad_music_filenames(m, c)
]

if not panels:
    print("No bad_music exclusions for the selected music types / conditions.")
else:
    fig, axes = plt.subplots(len(panels), 1, figsize=(15, 4.5 * len(panels)), squeeze=False)
    for ax, (music_type, condition) in zip(axes[:, 0], panels):
        incl_files = _group_df(music_type, condition, EXCLUSION_CATEGORIES)[
            SingleDataMetadata.FILENAME
        ].tolist()
        bad_files = bad_music_filenames(music_type, condition)
        incl_tags, sfreq = load_tags(incl_files)
        bad_tags, sfreq_b = load_tags(bad_files)
        sfreq = sfreq or sfreq_b

        def _window(sig):
            if TAG_SHOW_SEC is not None:
                sig = sig[: int(TAG_SHOW_SEC * sfreq)]
            return np.arange(len(sig)) / sfreq, sig

        for t in incl_tags:
            x, y = _window(t.tag_signal)
            ax.plot(x, y, lw=0.5, alpha=0.3, color="0.6")

        cmap = plt.cm.Reds
        n_bad = len(bad_tags)
        for k, t in enumerate(bad_tags):
            frac = 0.7 if n_bad == 1 else 0.55 + 0.4 * k / (n_bad - 1)
            x, y = _window(t.tag_signal)
            ax.plot(x, y, lw=1.3, color=cmap(frac), zorder=5,
                    label=f"{_pid_by_file.get(t.filename, t.filename)} (bad_music)")

        ax.set_title(
            f"TAG signal — {music_type.value} / {condition.value} "
            f"(included grey n={len(incl_tags)}, bad_music red n={n_bad})"
        )
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("TAG amplitude")
        ax.legend(loc="upper right", fontsize=8, framealpha=0.6)
        print(f"{music_type.value}/{condition.value}: bad_music = "
              f"{[_pid_by_file.get(f, f) for f in bad_files]}")

    plt.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(PLOTS_DIR / "bad_music_tag_comparison.png", dpi=150)
    plt.show()